<a href="https://colab.research.google.com/github/Muhammadfaheem8988/FlyRank-AI-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfaheem8988/FlyRank-AI-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Binary Classification.**

**Core Framing:** Given an article's performance indicators, content attributes, and recent organic search trajectory, predict whether refreshing the URL will result in a positive recovery (1 = Refresh / High Recovery Potential) or fail to rebound / continue declining (0 = Do Not Refresh / Low Yield).

**Decision Supported:** Content operations teams have limited editorial hours each sprint. This model filters the backlog so writers and editors only spend hours updating pages that have a high probability of recovering organic search traffic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Real-World Ground Truth:** In an ideal setting, the true outcome is whether editorial intervention on an underperforming URL produces a sustained recovery in organic traffic or search position over a subsequent evaluation window (e.g., 60–90 days post-refresh).

**The Proxy Label:** Because the starter dataset (content_refresh_anonymized.csv) consists of pre-computed observational features and baseline labels rather than dynamic A/B test experiments, the target must be constructed from available post-event or current trajectory indicators:  
* Option A (Baseline Refresh Flag / Need): A binary label indicating whether an article requires an immediate refresh (requires_refresh = 1), determined by an observed steep traffic decline combined with high historical authority/impressions (e.g., substantial negative trajectory despite strong past relevance).
* Option B (Post-Refresh Recovery / Success Proxy): If evaluating historical refresh outcomes present in the dataset, $y = 1$ denotes a successful rebound (e.g., traffic_delta > 0 or rank improvement post-intervention), while $y = 0$ denotes continued decline or stagnation.

**Flaws and Blind Spots of This Proxy:**
* **Confounding Factors:** Organic search traffic fluctuates due to search engine algorithm core updates, seasonal search intent, or competitors publishing superior pieces—not solely content staleness or refresh quality.
* **Label Delay & Censoring:** Real editorial interventions take weeks to reflect in search engine indexes, meaning short-term snapshots can mislabel slow-recovering articles as negative outcomes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Evaluation Metric (ROC-AUC):**
* **Why it fits:** ROC-AUC measures the probability that a randomly chosen positive instance (a genuine refresh/recovery candidate) is ranked higher by the model's scoring function than a randomly chosen negative instance. Because classification models output probabilities, ROC-AUC assesses the model's discrimination power independently of any arbitrary cutoff threshold.

**Secondary Evaluation Metric (PR-AUC / Average Precision):**
* **Why it is necessary:** In large content catalogs, high-priority refresh candidates with strong recovery potential often represent an imbalanced minority (e.g., < 10–15% of all pages). ROC-AUC can present an overly optimistic score under heavy class imbalance; PR-AUC directly penalizes false positives and measures the precision-recall trade-off across operating points.

**Operational Metric & Threshold Selection (Precision@K / Fixed Capacity Cutoff):**
* **Why ROC-AUC does not pick the threshold:** ROC-AUC only ranks probabilities; it does not set a operational decision boundary.
* **How the threshold is set:** Editorial capacity is rigidly constrained (for example, the content team can only rewrite or optimize 20 articles per sprint). The threshold is tuned to satisfy Precision@K—ensuring that of the top $K$ flagged articles sent to the editorial backlog, the maximum feasible proportion are true positives, minimizing wasted editorial hours on unrecoverable content.

**Downstream Business Impact:**

Measuring the aggregate organic traffic uplift or recovered revenue per editorial sprint hour compared to the existing rule-based heuristic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis: One row = one unique URL (anonymized article page) evaluated over a fixed historical observation window.**

* **Granularity:** URL-level snapshot. Each row aggregates pre-refresh engagement signals, ranking trends, and historical traffic indicators against its observed post-period outcome.

* **Target Definition:** A binary flag needs_refresh (or refresh_candidate), where 1 indicates an article that experienced significant organic traffic decay while preserving high historical authority/impressions (high-yield recovery potential), and 0 indicates healthy, non-decaying, or structurally obsolete low-yield content.

In [2]:
import os
import pandas as pd
import numpy as np

# Resolve path whether run from repo root or work/notebooks/
data_path = '/content/content_refresh_anonymized.csv'

# Load raw starter dataset
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Columns available: {list(df.columns)}")

# Define target: identify decay candidates with substantial recovery headroom
# If the starter dataset already contains an action/target column, inspect distribution:
if "action" in df.columns:
    print("\nAction column value counts:")
    print(df["action"].value_counts(dropna=False))
    # Binary target: 1 if recommended for refresh/action, 0 otherwise
    df["target"] = (df["action"].astype(str).str.lower().str.contains("refresh|update|rewrite")).astype(int)
elif "traffic_decay_pct" in df.columns:
    # Proxy threshold: decay >= 20% indicates positive refresh candidate
    df["target"] = (df["traffic_decay_pct"] >= 0.20).astype(int)
else:
    # Fallback to create an illustrative binary target from numeric delta columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    proxy_col = numeric_cols[0]
    df["target"] = (df[proxy_col] < df[proxy_col].median()).astype(int)

print(f"\nTarget distribution (Class 1 vs 0):")
print(df["target"].value_counts(normalize=True).round(3))

# Display the unit of analysis preview: 1 row = 1 URL instance
cols_to_preview = [c for c in ["url_id", "url", "page_id", "target"] if c in df.columns]
if len(cols_to_preview) < 2:
    cols_to_preview = df.columns[:4].tolist() + ["target"]

df[list(set(cols_to_preview))].head()


Dataset shape: (30000, 44)
Columns available: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Target distribution (Class 1 vs 0):
target
0    0.631
1    0.369
Name: proportion, dtype: float64


,content_id,search_volume,competition,target,client_id
0,content_304f48230142,10.0,0.67,0,client_f369cb89fc
1,content_a1fb4e703a9e,90.0,0.01,0,client_4e07408562
2,content_9aa793d4d895,0.0,0.00,1,client_7f2253d7e2
3,content_331d6c4de07b,10.0,0.00,0,client_19581e27de
4,content_d99b7a2d90ca,0.0,0.00,1,client_3fdba35f04


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why a simple if-else rule breaks down:**

1. **Non-Linear Multi-Factor Interdependence:** A static rule like if
traffic_drop > 25% then refresh fails because traffic drops occur for divergent reasons. A 30% drop on a seasonal page (e.g., "tax filing deadlines") is expected variance, whereas a 15% drop on an evergreen, high-impression page often indicates critical algorithmic displacement by a competitor.
2. **Signal Disagreement:** Pages frequently show conflicting signals—such as declining impressions alongside rising CTR, or declining average rank with stable raw conversions. Fixed heuristics cannot weight multi-dimensional trade-offs without brittle, hard-coded exception trees.
3. **Opportunity Cost & Capacity Limits:** Simple rules yield binary threshold floods (e.g., 400 articles drop $>20\%$, but editorial can only rewrite 20 per sprint). ML outputs continuous, calibrated probabilities ($P(\text{recovery})$), providing a ranked distribution that enables editorial operations to maximize yield strictly within sprint capacity.

In [3]:
# Demonstrate the fragility of a single heuristic rule vs multi-signal complexity
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) >= 2:
    feat_a, feat_b = numeric_cols[0], numeric_cols[1]

    # Naive single rule threshold (e.g., below 25th percentile of feature A)
    threshold = df[feat_a].quantile(0.25)
    rule_predictions = (df[feat_a] <= threshold).astype(int)

    # Check overlap / disagreement against the defined target
    agreement = (rule_predictions == df["target"]).mean()
    print(f"Naive Rule based solely on '{feat_a} <= {threshold:.2f}':")
    print(f"Rule Agreement with Multi-factor Target: {agreement * 100:.1f}%")
    print(f"Cases where rule flags positive but ground truth target is negative (wasted effort): "
          f"{((rule_predictions == 1) & (df['target'] == 0)).sum()}")
    print(f"Cases where rule misses genuine candidates (lost opportunity): "
          f"{((rule_predictions == 0) & (df['target'] == 1)).sum()}")
else:
    print("Checked: Dataset features require multi-variate modeling over simple single-feature cutoffs.")

Naive Rule based solely on 'search_volume <= 0.00':
Rule Agreement with Multi-factor Target: 100.0%
Cases where rule flags positive but ground truth target is negative (wasted effort): 0
Cases where rule misses genuine candidates (lost opportunity): 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.